# 随机森林回归（Python 3.11）

随机森林由多棵回归树组成。每棵树使用不同的自助采样（bootstrap）数据，并在每次节点分裂时只从随机选出的一部分特征中寻找最优切分。最后把所有树的预测值取平均，得到回归结果。

随机性主要来自两处：

1. **样本随机性**：每棵树从训练集中有放回地抽取同样数量的样本；
2. **特征随机性**：每个节点只考察随机选出的部分特征。

这两层随机化降低了各棵树之间的相关性，使集成后的模型更稳定。本 Notebook 使用 NumPy 从零实现，不依赖 `scikit-learn`，内容包括：

1. 回归树的递归分裂；
2. 随机森林的自助采样和随机特征选择；
3. 训练集、测试集和袋外（OOB）评估；
4. 特征重要性、新样本预测和残差诊断。

In [ ]:
# ============================================================
# 1. 导入依赖并设置绘图环境
# ============================================================
from __future__ import annotations

import sys
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 配置中文字体，并让负号在图中正常显示。
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print(f'Python 版本：{sys.version.split()[0]}')
print(f'NumPy 版本：{np.__version__}')
print(f'Pandas 版本：{pd.__version__}')

## 1. 准备数据

示例数据包含广告投入、门店面积、员工人数和客流量，目标变量为销售额。数据中加入非线性项和交互项，用来展示随机森林不依赖“目标变量必须是特征的线性组合”这一优势。

实际使用时，可以直接替换成自己的 CSV/Excel 数据。随机森林对特征量纲不敏感，但分类变量仍然需要先数值化。

In [ ]:
# ============================================================
# 2. 生成可重复的示例数据
# ============================================================
generator = np.random.default_rng(2026)
sample_count = 420

advertising = generator.uniform(10, 60, sample_count)
store_area = generator.uniform(50, 180, sample_count)
employees = generator.integers(3, 21, sample_count).astype(float)
customer_flow = generator.uniform(200, 1200, sample_count)

# 非线性项、交互项和随机噪声共同构成目标变量。
sales = (
    30
    + 6.2 * np.sqrt(advertising)
    + 0.42 * store_area
    + 2.1 * employees
    + 0.035 * customer_flow
    + 0.0009 * advertising * customer_flow
    + 0.08 * employees * store_area
    + generator.normal(0, 7.0, sample_count)
)

data = pd.DataFrame({
    '广告投入': advertising,
    '门店面积': store_area,
    '员工人数': employees,
    '客流量': customer_flow,
    '销售额': sales,
})

feature_names = ['广告投入', '门店面积', '员工人数', '客流量']
target_name = '销售额'

display(data.head())
print(f'样本数：{len(data)}，特征数：{len(feature_names)}')

### 替换为真实数据

```python
# data = pd.read_csv('your_data.csv')       # 也可以使用 read_excel
# feature_names = ['特征1', '特征2', '特征3']
# target_name = '目标变量'
# data = data.dropna(subset=feature_names + [target_name]).copy()
```

缺失值和异常值应结合业务含义处理。随机森林本身不能直接接收 `NaN`，上面的实现也会主动检查非有限值并报错。

## 2. 回归树

回归树在每个节点选择一个特征和阈值，把样本分成左右两部分。评价一次切分好坏时使用子节点残差平方和：

$$\mathrm{SSE}=\sum_{i\in\text{node}}(y_i-\bar y_{\text{node}})^2$$

最优切分使左右子节点的 SSE 之和最小。叶节点输出落在该区域全部训练样本的目标均值。

In [ ]:
# ============================================================
# 3. 回归树节点和回归树实现
# ============================================================
@dataclass
class TreeNode:
    """保存一个树节点需要的信息。

    叶节点没有 feature_index 和 threshold；内部节点同时拥有左右子树。
    """
    value: float                         # 当前节点样本目标值的平均数
    sample_count: int                   # 当前节点训练样本数
    impurity: float                     # 当前节点的残差平方和 SSE
    feature_index: int | None = None    # 分裂使用的特征下标
    threshold: float | None = None      # 左子树条件：feature <= threshold
    left: TreeNode | None = None
    right: TreeNode | None = None


class RegressionTree:
    """单棵回归树，仅在节点分裂时使用随机选出的特征子集。"""

    def __init__(self, max_depth=8, min_samples_split=5, min_samples_leaf=2,
                 max_features='sqrt', random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.rng = np.random.default_rng(random_state)
        self.root: TreeNode | None = None
        self.n_features_in_: int | None = None
        self.feature_importances_: np.ndarray | None = None

    @staticmethod
    def _squared_error(target):
        """计算一组目标值的残差平方和。"""
        if target.size == 0:
            return 0.0
        centered = target - target.mean()
        return float(np.sum(centered ** 2))

    def _resolve_max_features(self, feature_count):
        """把 sqrt、log2、比例或整数统一转换为实际考察的特征数。"""
        if self.max_features == 'sqrt':
            return max(1, int(np.sqrt(feature_count)))
        if self.max_features == 'log2':
            return max(1, int(np.log2(feature_count)))
        if isinstance(self.max_features, float):
            if not 0.0 < self.max_features <= 1.0:
                raise ValueError('浮点型 max_features 必须位于 (0, 1]')
            return max(1, int(np.ceil(self.max_features * feature_count)))
        if isinstance(self.max_features, (int, np.integer)):
            return min(feature_count, max(1, int(self.max_features)))
        raise ValueError('max_features 必须是 sqrt、log2、整数或 (0, 1] 的浮点数')

    def fit(self, features, target):
        """使用训练数据构建一棵回归树。"""
        feature_matrix = np.asarray(features, dtype=float)
        target_vector = np.asarray(target, dtype=float).reshape(-1)

        if feature_matrix.ndim != 2:
            raise ValueError('features 必须是二维矩阵')
        if feature_matrix.shape[0] != target_vector.size:
            raise ValueError('特征行数与目标变量长度不一致')
        if not np.isfinite(feature_matrix).all():
            raise ValueError('特征中包含 NaN 或无穷大')
        if not np.isfinite(target_vector).all():
            raise ValueError('目标变量中包含 NaN 或无穷大')

        self.n_features_in_ = feature_matrix.shape[1]
        self._actual_max_features = self._resolve_max_features(self.n_features_in_)
        self.feature_importances_ = np.zeros(self.n_features_in_, dtype=float)

        # 根节点保存训练集总样本数，用于计算整棵树加权后的特征重要性。
        self._root_sample_count = feature_matrix.shape[0]
        self.root = self._grow(feature_matrix, target_vector, depth=0)

        # 把所有特征重要性归一化到总和为 1。
        total_importance = self.feature_importances_.sum()
        if total_importance > 0:
            self.feature_importances_ /= total_importance
        return self

    def _grow(self, features, target, depth):
        """递归生成树节点。"""
        sample_count, feature_count = features.shape
        node_value = float(target.mean())
        node_impurity = self._squared_error(target)

        # 达到深度、样本数或纯节点条件时停止分裂。
        if (
            depth >= self.max_depth
            or sample_count < self.min_samples_split
            or node_impurity <= np.finfo(float).eps
        ):
            return TreeNode(node_value, sample_count, node_impurity)

        # 随机森林与单棵决策树的关键区别：每个节点只看随机特征子集。
        candidate_count = min(self._actual_max_features, feature_count)
        candidate_features = self.rng.choice(
            feature_count, size=candidate_count, replace=False
        )

        best_sse = np.inf
        best_feature = None
        best_threshold = None

        for feature_index in candidate_features:
            values = features[:, feature_index]
            order = np.argsort(values, kind='mergesort')
            sorted_values = values[order]
            sorted_target = target[order]

            # 只有相邻特征值不同才存在有效阈值。
            distinct = sorted_values[1:] > sorted_values[:-1]
            left_counts = np.arange(1, sample_count, dtype=int)

            # 左右两侧都必须满足最小叶节点样本数。
            valid_splits = (
                distinct
                & (left_counts >= self.min_samples_leaf)
                & ((sample_count - left_counts) >= self.min_samples_leaf)
            )
            if not np.any(valid_splits):
                continue

            cumulative_sum = np.cumsum(sorted_target)
            cumulative_square_sum = np.cumsum(sorted_target ** 2)

            left_sum = cumulative_sum[:-1]
            left_square_sum = cumulative_square_sum[:-1]
            total_sum = cumulative_sum[-1]
            total_square_sum = cumulative_square_sum[-1]

            right_counts = sample_count - left_counts
            left_sse = left_square_sum - left_sum ** 2 / left_counts
            right_sse = (
                (total_square_sum - left_square_sum)
                - (total_sum - left_sum) ** 2 / right_counts
            )

            # 浮点误差可能产生极小的负数，因此裁剪到 0。
            split_sse = np.maximum(left_sse, 0.0) + np.maximum(right_sse, 0.0)
            split_sse = np.where(valid_splits, split_sse, np.inf)

            best_position = int(np.argmin(split_sse))
            if split_sse[best_position] < best_sse:
                best_sse = float(split_sse[best_position])
                best_feature = int(feature_index)
                best_threshold = float(
                    (sorted_values[best_position] + sorted_values[best_position + 1]) / 2
                )

        # 找不到有效切分，或者切分没有降低误差时，生成叶节点。
        if best_feature is None or node_impurity - best_sse <= 1e-12:
            return TreeNode(node_value, sample_count, node_impurity)

        # 使用该节点占总样本的比例加权误差下降量。
        importance_gain = (
            sample_count / self._root_sample_count * (node_impurity - best_sse)
        )
        self.feature_importances_[best_feature] += importance_gain

        left_mask = features[:, best_feature] <= best_threshold
        right_mask = ~left_mask

        left_child = self._grow(features[left_mask], target[left_mask], depth + 1)
        right_child = self._grow(features[right_mask], target[right_mask], depth + 1)

        return TreeNode(
            value=node_value,
            sample_count=sample_count,
            impurity=node_impurity,
            feature_index=best_feature,
            threshold=best_threshold,
            left=left_child,
            right=right_child,
        )

    def _predict_one(self, row, node):
        """把一个样本从根节点传递到叶节点。"""
        while node.feature_index is not None:
            if row[node.feature_index] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.value

    def predict(self, features):
        """预测多个样本。"""
        if self.root is None:
            raise RuntimeError('模型尚未训练，请先调用 fit')

        feature_matrix = np.asarray(features, dtype=float)
        if feature_matrix.ndim == 1:
            feature_matrix = feature_matrix.reshape(1, -1)
        if feature_matrix.shape[1] != self.n_features_in_:
            raise ValueError('预测样本的特征数必须与训练时一致')
        if not np.isfinite(feature_matrix).all():
            raise ValueError('预测样本中包含 NaN 或无穷大')

        return np.array([
            self._predict_one(row, self.root) for row in feature_matrix
        ])

## 3. 随机森林

每棵树使用一份自助采样数据集。由于是有放回抽样，平均约 36.8% 的原始样本不会出现在某棵树的训练集中，这部分样本称为该树的袋外样本（OOB）。利用 OOB 预测可以形成一个近似免费的验证结果。

需要注意：OOB 分数适合作为模型稳定性的参考，但不能替代严格保留的最终测试集。

In [ ]:
# ============================================================
# 4. 随机森林和评估指标
# ============================================================
class RandomForestRegressor:
    """由多棵随机回归树平均组成的随机森林。"""

    def __init__(self, n_estimators=100, max_depth=8, min_samples_split=5,
                 min_samples_leaf=2, max_features='sqrt', random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.trees_ = []
        self.n_features_in_: int | None = None

    def fit(self, features, target):
        """训练全部回归树，并计算 OOB 预测和特征重要性。"""
        feature_matrix = np.asarray(features, dtype=float)
        target_vector = np.asarray(target, dtype=float).reshape(-1)

        if feature_matrix.ndim != 2:
            raise ValueError('features 必须是二维矩阵')
        if feature_matrix.shape[0] != target_vector.size:
            raise ValueError('特征行数与目标变量长度不一致')
        if self.n_estimators < 1:
            raise ValueError('n_estimators 必须至少为 1')

        sample_count, feature_count = feature_matrix.shape
        self.n_features_in_ = feature_count

        # 从同一个主随机数生成器派生每棵树的随机种子，保证整体结果可复现。
        main_rng = np.random.default_rng(self.random_state)
        tree_seeds = main_rng.integers(0, np.iinfo(np.int32).max, self.n_estimators)

        oob_sum = np.zeros(sample_count, dtype=float)
        oob_count = np.zeros(sample_count, dtype=int)
        self.trees_ = []

        for tree_seed in tree_seeds:
            # 有放回抽样：某些样本出现多次，另一些样本一次也不出现。
            bootstrap_indices = main_rng.integers(0, sample_count, sample_count)
            bootstrap_mask = np.zeros(sample_count, dtype=bool)
            bootstrap_mask[bootstrap_indices] = True
            oob_mask = ~bootstrap_mask

            tree = RegressionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=self.max_features,
                random_state=int(tree_seed),
            )
            tree.fit(
                feature_matrix[bootstrap_indices],
                target_vector[bootstrap_indices],
            )
            self.trees_.append(tree)

            # 只使用没有进入该树训练集的样本计算 OOB 预测。
            if np.any(oob_mask):
                tree_oob_prediction = tree.predict(feature_matrix[oob_mask])
                oob_sum[oob_mask] += tree_oob_prediction
                oob_count[oob_mask] += 1

        # 每个 OOB 样本对可能被多棵树预测，因此取平均值。
        self.oob_prediction_ = np.full(sample_count, np.nan, dtype=float)
        covered = oob_count > 0
        self.oob_prediction_[covered] = oob_sum[covered] / oob_count[covered]

        # 对所有树的特征重要性取平均。
        self.feature_importances_ = np.mean([
            tree.feature_importances_ for tree in self.trees_
        ], axis=0)
        total_importance = self.feature_importances_.sum()
        if total_importance > 0:
            self.feature_importances_ /= total_importance

        return self

    def predict(self, features):
        """所有树的预测值取算术平均。"""
        if not self.trees_:
            raise RuntimeError('模型尚未训练，请先调用 fit')

        feature_matrix = np.asarray(features, dtype=float)
        if feature_matrix.ndim == 1:
            feature_matrix = feature_matrix.reshape(1, -1)
        if feature_matrix.shape[1] != self.n_features_in_:
            raise ValueError('预测样本的特征数必须与训练时一致')

        tree_predictions = np.vstack([
            tree.predict(feature_matrix) for tree in self.trees_
        ])
        return tree_predictions.mean(axis=0)


def regression_metrics(y_true, y_pred, feature_count):
    """计算回归模型常用指标。"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    residuals = y_true - y_pred

    total_sum_squares = np.sum((y_true - y_true.mean()) ** 2)
    residual_sum_squares = np.sum(residuals ** 2)

    r_squared = 1.0 - residual_sum_squares / total_sum_squares
    sample_count = y_true.size
    adjusted_r_squared = 1.0 - (
        (1.0 - r_squared)
        * (sample_count - 1)
        / (sample_count - feature_count - 1)
    )
    rmse = np.sqrt(np.mean(residuals ** 2))
    mae = np.mean(np.abs(residuals))

    return {
        'R²': r_squared,
        '调整R²': adjusted_r_squared,
        'RMSE': rmse,
        'MAE': mae,
    }

## 4. 划分数据并训练模型

使用固定随机种子把样本划分为 80% 训练集和 20% 测试集。测试集在整个训练过程中完全不参与树结构学习，只用于最终评估。

主要超参数：

- `n_estimators`：树的数量。增加通常能提高稳定性，但计算量近似线性增加；
- `max_depth`：树的最大深度。越深越容易拟合复杂关系，也更可能过拟合；
- `min_samples_leaf`：叶节点最少样本数，能抑制孤立样本导致的高方差；
- `max_features='sqrt'`：每次分裂只从约 $\sqrt{p}$ 个特征中搜索，是随机森林降低树相关性的关键参数。

In [ ]:
# ============================================================
# 5. 按 80%/20% 划分训练集和测试集
# ============================================================
train_rng = np.random.default_rng(7)
shuffled_indices = train_rng.permutation(len(data))
test_size = int(round(len(data) * 0.20))
test_indices = shuffled_indices[:test_size]
train_indices = shuffled_indices[test_size:]

X_all = data[feature_names].to_numpy(dtype=float)
y_all = data[target_name].to_numpy(dtype=float)

X_train = X_all[train_indices]
X_test = X_all[test_indices]
y_train = y_all[train_indices]
y_test = y_all[test_indices]

print(f'训练集：{X_train.shape[0]} 条，测试集：{X_test.shape[0]} 条')

# ============================================================
# 训练随机森林
# max_features='sqrt' 表示每个节点随机考察约 sqrt(4)=2 个特征。
# ============================================================
forest = RandomForestRegressor(
    n_estimators=120,
    max_depth=9,
    min_samples_split=6,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
)

start_time = time.perf_counter()
forest.fit(X_train, y_train)
training_seconds = time.perf_counter() - start_time

train_prediction = forest.predict(X_train)
test_prediction = forest.predict(X_test)

# OOB 只针对训练集，因此先单独计算训练集 OOB 指标。
oob_valid = np.isfinite(forest.oob_prediction_)
oob_metrics = regression_metrics(
    y_train[oob_valid],
    forest.oob_prediction_[oob_valid],
    len(feature_names),
)
train_metrics = regression_metrics(y_train, train_prediction, len(feature_names))
test_metrics = regression_metrics(y_test, test_prediction, len(feature_names))

print(f'训练耗时：{training_seconds:.2f} 秒')
print(f'获得 OOB 预测的训练样本：{oob_valid.sum()} / {len(y_train)}')

metrics_table = pd.DataFrame({
    '数据集': ['训练集', 'OOB（训练集）', '测试集'],
    'R²': [train_metrics['R²'], oob_metrics['R²'], test_metrics['R²']],
    '调整R²': [
        train_metrics['调整R²'],
        oob_metrics['调整R²'],
        test_metrics['调整R²'],
    ],
    'RMSE': [
        train_metrics['RMSE'],
        oob_metrics['RMSE'],
        test_metrics['RMSE'],
    ],
    'MAE': [
        train_metrics['MAE'],
        oob_metrics['MAE'],
        test_metrics['MAE'],
    ],
})

display(metrics_table.round(4))

## 5. 特征重要性与新样本预测

下面的重要性来自各次分裂造成的加权误差下降。它表示“特征对模型预测的贡献程度”，不等同于因果关系，也可能高估连续特征或高基数特征。

随机森林只能在新样本附近做分段常数的平均预测，对训练范围之外的外推通常不可靠。

In [ ]:
# ============================================================
# 6. 特征重要性和新样本预测
# ============================================================
importance_table = pd.DataFrame({
    '特征': feature_names,
    '重要性': forest.feature_importances_,
}).sort_values('重要性', ascending=False)

display(importance_table.round(4))

new_samples = pd.DataFrame([
    [25.0, 90.0, 8.0, 500.0],
    [45.0, 130.0, 14.0, 900.0],
], columns=feature_names)

new_samples['预测销售额'] = forest.predict(new_samples)
display(new_samples.round(2))

## 6. 模型诊断

随机森林没有线性回归那样的系数置信区间，通常通过以下方式判断模型质量：

1. 测试集 $R^2$ 是否明显低于训练集，以判断过拟合；
2. 真实值和预测值是否接近 45° 线；
3. 残差是否围绕 0 随机分布；
4. 特征重要性是否符合业务认知；
5. 使用交叉验证或多个随机种子确认结果稳定性。

In [ ]:
# ============================================================
# 7. 绘制测试集预测、残差和特征重要性
# ============================================================
test_residuals = y_test - test_prediction

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

# 左图：点越接近红色虚线，说明预测越准确。
axes[0].scatter(y_test, test_prediction, alpha=0.75, edgecolors='white')
line_min = min(y_test.min(), test_prediction.min())
line_max = max(y_test.max(), test_prediction.max())
axes[0].plot([line_min, line_max], [line_min, line_max],
             color='red', linestyle='--', label='完美预测线')
axes[0].set_xlabel('真实值')
axes[0].set_ylabel('预测值')
axes[0].set_title('测试集：真实值 vs 预测值')
axes[0].legend()

# 中图：残差应围绕 0 随机分布，不应出现明显漏斗形。
axes[1].scatter(test_prediction, test_residuals,
               alpha=0.75, edgecolors='white')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('预测值')
axes[1].set_ylabel('残差')
axes[1].set_title('测试集残差')

# 右图：按重要性从大到小绘制水平条形图。
plot_importance = importance_table.sort_values('重要性')
axes[2].barh(plot_importance['特征'], plot_importance['重要性'], color='#4C72B0')
axes[2].set_xlabel('归一化重要性')
axes[2].set_title('随机森林特征重要性')

plt.tight_layout()
plt.show()

## 7. 使用建议

1. **先检查数据泄漏**：目标变量未来信息、样本重复或分组信息混入特征，会制造虚假的高分。
2. **训练集与测试集要独立划分**：时间序列数据应按时间切分，不能随机打乱未来和过去。
3. **不要只比较一个随机种子**：不同种子会改变森林结构，可用多次运行或交叉验证评估稳定性。
4. **根据业务成本选择指标**：RMSE 对大误差更敏感，MAE 更容易直观解释。
5. **重要不等于因果**：要判断变量影响机制，仍需结合实验设计、混杂因素分析和领域知识。